1. Store master table — only needs to run once, but safe to leave in since it's a simple overwrite

In [0]:
from pyspark.sql import Row
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, row_number, desc

store_master = [
    Row(store_id="ST001", city="San Jose", region="West", category_focus="Electronics"),
    Row(store_id="ST002", city="Los Angeles", region="West", category_focus="Apparel"),
    Row(store_id="ST003", city="Seattle", region="West", category_focus="Outdoor Gear"),
    Row(store_id="ST004", city="Denver", region="Mountain", category_focus="Outdoor Gear"),
    Row(store_id="ST005", city="Phoenix", region="Mountain", category_focus="Home & Garden"),
    Row(store_id="ST006", city="Chicago", region="Midwest", category_focus="Apparel"),
    Row(store_id="ST007", city="Minneapolis", region="Midwest", category_focus="Home & Garden"),
    Row(store_id="ST008", city="Dallas", region="South", category_focus="Electronics"),
    Row(store_id="ST009", city="Houston", region="South", category_focus="Home & Garden"),
    Row(store_id="ST010", city="Miami", region="South", category_focus="Apparel"),
    Row(store_id="ST011", city="Atlanta", region="South", category_focus="Electronics"),
    Row(store_id="ST012", city="New York", region="Northeast", category_focus="Apparel"),
    Row(store_id="ST013", city="Boston", region="Northeast", category_focus="Outdoor Gear"),
]

store_master_df = spark.createDataFrame(store_master)
store_master_df.write.format("delta").mode("overwrite") \
    .saveAsTable("retail_demo.retail_gold.store_master")

2. Build the demand signal scoring logic:

In [0]:

silver_df = spark.table("retail_demo.retail_silver.weather_clean")
master_df = spark.table("retail_demo.retail_gold.store_master").select("store_id", "category_focus")


3. Scoring function (applied identically to both current and history so logic never drifts apart)

In [0]:
def score_demand(df):
    joined = df.join(master_df, on="store_id", how="left")
    return joined.withColumns({
        "rain_gear_demand": when(col("precipitation_in") > 0.1, "High")
            .when(col("precipitation_in") > 0, "Medium")
            .otherwise("Low"),
        "ac_cooling_demand": when(col("temperature_f") >= 90, "High")
            .when(col("temperature_f") >= 75, "Medium")
            .otherwise("Low"),
        "outdoor_event_risk": when((col("precipitation_in") > 0.1) | (col("wind_speed_mph") > 25), "High Risk")
            .otherwise("Low Risk"),
        "staffing_recommendation": when((col("temperature_f") >= 90) | (col("precipitation_in") > 0.1), "Increase Staff")
            .otherwise("Standard Staffing")
    })

4. CURRENT table — latest reading per store, overwrite

In [0]:
latest_window = Window.partitionBy("store_id").orderBy(desc("observation_time"))

latest_silver_df = silver_df.withColumn("rn", row_number().over(latest_window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

current_df = score_demand(latest_silver_df)

current_df.write.format("delta").mode("overwrite") \
    .saveAsTable("retail_demo.retail_gold.demand_signal_current")


5. HISTORY table — full time series, MERGE to avoid duplicate (store_id, observation_time) rows

In [0]:
history_df = score_demand(silver_df)

if not spark.catalog.tableExists("retail_demo.retail_gold.demand_signal_history"):
    history_df.write.format("delta").saveAsTable("retail_demo.retail_gold.demand_signal_history")
    print("Created history table")
else:
    history_df.createOrReplaceTempView("history_updates")
    spark.sql("""
        MERGE INTO retail_demo.retail_gold.demand_signal_history AS target
        USING history_updates AS source
        ON target.store_id = source.store_id AND target.observation_time = source.observation_time
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print("Merged into history table")

6. Sanity checks

In [0]:
print("Current table row count (should stay at 13):",
      spark.table("retail_demo.retail_gold.demand_signal_current").count())
print("History table row count (should grow over time):",
      spark.table("retail_demo.retail_gold.demand_signal_history").count())

7. New forecast demand signal 

In [0]:
forecast_silver_df = spark.table("retail_demo.retail_silver.weather_forecast_clean")
master_df = spark.table("retail_demo.retail_gold.store_master").select("store_id", "category_focus")

forecast_joined = forecast_silver_df.join(master_df, on="store_id", how="left")

forecast_gold_df = forecast_joined.withColumn(
    "rain_gear_demand",
    when(col("precip_prob_pct") >= 60, "High")
    .when(col("precip_prob_pct") >= 30, "Medium")
    .otherwise("Low")
).withColumn(
    "ac_cooling_demand",
    when(col("temp_max_f") >= 90, "High")
    .when(col("temp_max_f") >= 75, "Medium")
    .otherwise("Low")
).withColumn(
    "staffing_recommendation",
    when((col("temp_max_f") >= 90) | (col("precip_prob_pct") >= 60), "Increase Staff")
    .otherwise("Standard Staffing")
)

# Overwrite each run — forecast for a given date can change as it gets closer, so we always
# want the latest read for every date, not an accumulating history of forecast revisions
forecast_gold_df.write.format("delta").mode("overwrite") \
    .saveAsTable("retail_demo.retail_gold.demand_signal_forecast")